In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
cap = cv2.VideoCapture('Robots.mp4')
ret, frame = cap.read()
frame = cv2.resize(frame, None, fx=0.5, fy=0.5)
prvs_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
hsv = np.zeros_like(frame)
hsv[..., 1] = 255
frame_list=[]
while cap.isOpened(): 
    ret, frame = cap.read()
    if not ret:
        print('No frames grabbed!')
        break
    frame = cv2.resize(frame, None, fx=0.5, fy=0.5)
    gray_now =cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(prvs_gray, gray_now, None, 0.5, 3, 20, 3, 5, 1.5, 0)
    mag, ang = cv2.cartToPolar(flow[:,:,0], flow[:,:,1]) # Retrieving the magnitude and angle of every pixel
    
    hsv[..., 0] = ang*180/np.pi/2 # hue calculation
    hsv[..., 1] = 255 # Saturation = maximum, so direction is clearly represented by colour
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX) # Value, magnitude of motion 
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    frame_list.append(bgr)
    
    prvs_gray=gray_now

for frame in frame_list:
    cv2.imshow('image', frame)
    wait = cv2.waitKey(4) & 0xff
    if wait == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
# documentation: https://docs.opencv.org/4.13.0/d4/dee/tutorial_optical_flow.html

No frames grabbed!


# Explaination 
Here, dense optical flow is utilized on the video. To calculate dense optical flow at each frame, we take the current and previous grayscaled image. In terms of the parameters, a classic pyramid image scale where each layer is twice as small as the last one was used. 3 pyramid layers were used, based off standard usage in documentation. 20 was found to be a good window size such that it balances blurred motion with fast motion detection. The frames were also resized to half to reduce the compute time. To give smooth playback, the dense optical flow was first computed on the video and the frames were saved in a list. Then the frames are played after to reduce lag from real-time computation.  

In [ ]:
#dense
#Now we are going to print the motion vectors in a mask.
#This mask is mixed with the video to create a visual trail of the robots' movement.
cap = cv2.VideoCapture("Robots.mp4")
ret, frame = cap.read()
b,g,r = cv2.split(frame)
img = cv2.merge([r,g,b])
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
hsv=np.zeros_like(frame)
hsv[...,1]=255

mask = np.zeros_like(img)
while True:
    ret2, frame2 = cap.read()
    if not ret2:
        break
    b,g,r = cv2.split(frame2)
    img2 = cv2.merge([r,g,b])
    gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(gray, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[:, :, 0], flow[:, :, 1])
    hsv[...,0]=ang*180/np.pi/2
    hsv[...,1]=255
    hsv[...,2]=cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    bgr=cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    step = 8
    for y in range(0, gray.shape[0], step):
        for x in range(0, gray.shape[1], step):
            dx = flow[y, x, 0]
            dy = flow[y, x, 1]
            if mag[y, x] > 4:      # Solo si hay movimiento
                x2 = int(x + dx)
                y2 = int(y + dy)
                cv2.line(mask, (x, y), (x2, y2), (255, 255, 255), 1)

    gray = gray2.copy()
    result = cv2.add(img2, mask)
    cv2.imshow("Dense Optical Flow", cv2.cvtColor(result, cv2.COLOR_RGB2BGR))
    cv2.waitKey(2) 
plt.imshow(result) 
cap.release()
cv2.destroyAllWindows()

In [ ]:
# EXPLANATION
# This is also a dense approach, so the program detects the motion of all pixels.
# To avoid drawing too many lines, an 8-pixel step is used.
# The motion lines are drawn on a black mask, which is then merged with the video frames and displayed.

## Challenge
Track only one running robot